In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv ('202410DatasetEVOfficeParking_v0.csv')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['start_datetime'] = pd.to_datetime(df['start_datetime'], dayfirst = True)
df['end_datetime'] = pd.to_datetime(df['end_datetime'], dayfirst = True )

df['duration_min'] = (df['end_datetime'] - df['start_datetime']).dt.total_seconds() / 60
df['avg_power_kw'] = df['total_energy'] / (df['duration_min'] / 60)

In [ ]:
df.head()

In [ ]:
import numpy as np

df['label'] = 0

attack_df = df.sample(n=3000, random_state=42).copy()

n = len(attack_df)
attack_types = np.random.choice(['fdi', 'ddos', 'mitm'], size=n)

for i, atype in zip(attack_df.index, attack_types):
    if atype == 'fdi':
        attack_df.loc[i, 'total_energy'] *= np.random.uniform(1.5, 3)       # pehle 5-10x tha
    elif atype == 'ddos':
        attack_df.loc[i, 'duration_min'] *= np.random.uniform(0.3, 0.5)     # pehle 0.1-1 min fixed tha
        attack_df.loc[i, 'avg_power_kw'] = attack_df.loc[i, 'total_energy'] / (attack_df.loc[i, 'duration_min'] / 60)
    elif atype == 'mitm':
        attack_df.loc[i, 'avg_power_kw'] *= np.random.uniform(-0.5, 0.5)    # pehle -3 se -1 tha (poora negative)

attack_df['label'] = 1

final_df = pd.concat([df, attack_df], ignore_index=True)
final_df['label'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

features = ['total_energy', 'duration_min', 'avg_power_kw']
X = final_df[features]
y = final_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

# Logistic Regression ko scaled data chahiye hota (Random Forest ko zaroorat nahi thi)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

results = []

for name, m in models.items():
    if name == 'Logistic Regression':
        m.fit(X_train_scaled, y_train)
        pred = m.predict(X_test_scaled)
    else:
        m.fit(X_train, y_train)
        pred = m.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred)
    })

comparison_df = pd.DataFrame(results)
comparison_df

In [ ]:
import matplotlib.pyplot as plt

# Graph 1: Model comparison bar chart
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1 Score']].plot(
    kind='bar', figsize=(10, 6), colormap='viridis'
)
plt.title('Model Comparison: Accuracy, Precision, Recall, F1 Score')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()

In [ ]:
# Graph 2: Feature importance 
importances = model.feature_importances_
feat_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(feat_importance_df['Feature'], feat_importance_df['Importance'], color='teal')
plt.xlabel('Importance Score')
plt.title('Feature Importance — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()

feat_importance_df

In [ ]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(contamination=0.05, random_state=42)
iso_model.fit(X_train)

iso_pred = iso_model.predict(X_test)
iso_pred = [1 if p == -1 else 0 for p in iso_pred]

print("Accuracy:", accuracy_score(y_test, iso_pred))
print("Precision:", precision_score(y_test, iso_pred))
print("Recall:", recall_score(y_test, iso_pred))
print("F1 Score:", f1_score(y_test, iso_pred))

In [ ]:
import joblib
joblib.dump(model, 'rf_model.pkl')